<div align="center">

# 🚸 Detecção de Faixa de Pedestre com YOLO

### Comparação entre YOLOv5 · YOLOv8 · YOLO11 · YOLO12

<a href="https://colab.research.google.com/github/DaviPRabelo/CrossWalkDetection/blob/main/CrosswalkDetection.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"></a>
<img src="https://img.shields.io/badge/Python-3.12-3776AB?logo=python&logoColor=white" alt="Python">
<img src="https://img.shields.io/badge/Ultralytics-8.3.229-0B23A9" alt="Ultralytics">
<img src="https://img.shields.io/badge/Dataset-Roboflow-6706CE" alt="Roboflow">

</div>

---

Quatro gerações do YOLO treinadas na **mesma versão do dataset**, com os **mesmos hiperparâmetros** e avaliadas no **mesmo split de teste**.

| | Modelo | Pesos iniciais | Formato do dataset | Código de treino |
|:-:|---|---|---|---|
| 🟦 | **YOLOv5n** | `yolov5n.pt` | YOLOv5 PyTorch (`yolov5`) | repositório original [`ultralytics/yolov5`](https://github.com/ultralytics/yolov5) |
| 🟩 | **YOLOv8n** | `yolov8n.pt` | `yolov8` | pacote `ultralytics` |
| 🟧 | **YOLO11n** | `yolo11n.pt` | `yolov11` | pacote `ultralytics` |
| 🟥 | **YOLO12n** | `yolo12n.pt` | `yolov12` | pacote `ultralytics` |

Todos partem de pesos pré-treinados no COCO.

## 📑 Sumário

1. [⚙️ Configuração](#configuracao)
2. [🖥️ Ambiente](#ambiente)
3. [🟦 YOLOv5n](#yolov5)
4. [🟩 YOLOv8n](#yolov8)
5. [🟧 YOLO11n](#yolo11)
6. [🟥 YOLO12n](#yolo12)
7. [📊 Comparação](#comparacao)
8. [🔍 Inferência](#inferencia)

> 💡 **Antes de começar:** no Colab, vá em `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU**.
>
> 🧩 **Cada modelo é independente.** Depois de rodar as seções 1 e 2, você pode rodar só a seção do modelo que quiser. As métricas de cada um ficam salvas no Drive, e a seção 7 junta o que já tiver sido treinado.

<a id="configuracao"></a>
# ⚙️ 1. Configuração

> 🔐 **A chave da Roboflow não fica no código.** Cadastre-a nos **Secrets** do Colab (ícone 🔑 na barra lateral) com o nome `ROBOFLOW_API_KEY` e ative o acesso para este notebook.

In [ ]:
# ── Dataset (Roboflow) ─────────────────────────────────────────────
# https://app.roboflow.com/workspace-hjntk/crossalkzebra-sdro0
RF_WORKSPACE = "workspace-hjntk"
RF_PROJECT   = "crossalkzebra-sdro0"
RF_VERSION   = 3

# ── Hiperparâmetros (iguais para todos os modelos) ─────────────────
EPOCHS = 50
IMGSZ  = 640
BATCH  = 16
SEED   = 0

# ── Onde salvar os resultados ──────────────────────────────────────
USE_DRIVE = True                                     # False → só na sessão do Colab
DRIVE_DIR = "/content/drive/MyDrive/Capacitacao/faixa-yolo"

<a id="ambiente"></a>
# 🖥️ 2. Ambiente

In [ ]:
# @title GPU disponível
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# @title Instalar dependências
# O requirements.txt do YOLOv5 instala a versão mais nova do ultralytics. Se isso acontecer
# depois do import, a sessão fica com duas versões misturadas (erro "No module named
# 'ultralytics.hub'"). Por isso tudo é instalado aqui, antes de qualquer import, e o
# ultralytics é fixado por último.
YOLOV5_DIR = "/content/yolov5"
!test -d {YOLOV5_DIR} || git clone -q https://github.com/ultralytics/yolov5 {YOLOV5_DIR}
%pip install -q -r {YOLOV5_DIR}/requirements.txt
%pip install -q "ultralytics==8.3.229" roboflow

In [ ]:
# @title Importações e estilo dos gráficos
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import yaml
import ultralytics
from google.colab import userdata
from roboflow import Roboflow
from ultralytics import YOLO
from IPython.display import Image, Markdown, display

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

MODEL_ORDER = ["YOLOv5n", "YOLOv8n", "YOLO11n", "YOLO12n"]
COLORS = {"YOLOv5n": "#1f77b4", "YOLOv8n": "#2ca02c", "YOLO11n": "#ff7f0e", "YOLO12n": "#d62728"}
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titleweight": "bold",
})

assert ultralytics.__version__ == "8.3.229", (
    f"ultralytics {ultralytics.__version__} carregado; reinicie a sessão "
    "(Ambiente de execução → Reiniciar sessão) e rode a partir da seção 2."
)
ultralytics.checks()

In [ ]:
# @title Pasta de resultados
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(DRIVE_DIR)
else:
    OUT_DIR = Path("/content/faixa-yolo")

RUNS_DIR = OUT_DIR / "runs"
METRICS_DIR = OUT_DIR / "metricas"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
DATA = {}   # nome do modelo → caminho do data.yaml

display(Markdown(f"📁 Resultados serão salvos em **`{OUT_DIR}`**"))

In [ ]:
# @title 🧰 Funções auxiliares
def preparar_dataset(location):
    # O data.yaml da Roboflow usa caminhos relativos (../train/images) que às vezes
    # não são encontrados. Reescrevemos com caminhos absolutos.
    location = Path(location)
    data_yaml = location / "data.yaml"
    cfg = yaml.safe_load(data_yaml.read_text())
    cfg["path"]  = str(location)
    cfg["train"] = "train/images"
    cfg["val"]   = "valid/images"
    if (location / "test" / "images").exists():
        cfg["test"] = "test/images"
    else:
        cfg.pop("test", None)
    data_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return data_yaml


def split_avaliacao(data_yaml):
    return "test" if "test" in yaml.safe_load(Path(data_yaml).read_text()) else "val"


def _ler_caixas(label_file):
    # Aceita caixa (cls cx cy w h) ou polígono (cls x1 y1 x2 y2 ...).
    caixas = []
    if not label_file.exists():
        return caixas
    for line in label_file.read_text().split("\n"):
        v = line.split()
        if len(v) < 5:
            continue
        c, xy = int(v[0]), list(map(float, v[1:]))
        if len(xy) == 4:
            cx, cy, w, h = xy
            caixas.append((c, cx - w / 2, cy - h / 2, w, h))
        else:
            xs, ys = xy[0::2], xy[1::2]
            caixas.append((c, min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)))
    return caixas


def resumo_dataset(data_yaml, n_amostras=4):
    root = Path(data_yaml).parent
    names = yaml.safe_load(Path(data_yaml).read_text())["names"]

    def count(split, sub):
        d = root / split / sub
        return len(list(d.glob("*"))) if d.exists() else 0

    splits = {"train": "🏋️ treino", "valid": "🧐 validação", "test": "🧪 teste"}
    tabela = pd.DataFrame(
        {"Imagens": [count(s, "images") for s in splits],
         "Rótulos": [count(s, "labels") for s in splits]},
        index=list(splits.values()),
    )
    tabela.loc["Σ total"] = tabela.sum()
    display(Markdown(f"📂 `{root}`  \n**Classes:** {', '.join(f'`{c}`' for c in names)}"))
    display(tabela.style.format("{:,}"))

    imgs = sorted((root / "train" / "images").glob("*"))
    imgs = imgs[:: max(1, len(imgs) // n_amostras)][:n_amostras]
    fig, axes = plt.subplots(1, len(imgs), figsize=(4.5 * len(imgs), 3.5), squeeze=False)
    for ax, p in zip(axes.flat, imgs):
        img = plt.imread(p)
        h, w = img.shape[:2]
        ax.imshow(img)
        for c, x, y, bw, bh in _ler_caixas(root / "train" / "labels" / f"{p.stem}.txt"):
            ax.add_patch(patches.Rectangle((x * w, y * h), bw * w, bh * h, fill=False, lw=2, ec="#00e676"))
            ax.text(x * w, y * h - 4, names[c], color="black", fontsize=8,
                    bbox=dict(fc="#00e676", ec="none", pad=1))
        ax.axis("off")
    fig.suptitle("Amostras do treino com anotações", fontweight="bold")
    plt.tight_layout()
    plt.show()


def ja_treinado(name):
    best = RUNS_DIR / name / "weights" / "best.pt"
    if best.exists():
        display(Markdown(f"✅ **{name}** já treinado (`{best}`), pulando o treino."))
        return True
    return False


def mostrar_metricas(name, m):
    display(Markdown(
        f"#### {name} — split `{m['Split']}`\n\n"
        "| Precisão | Revocação | mAP50 | mAP50-95 | Inferência | Parâmetros |\n"
        "|:-:|:-:|:-:|:-:|:-:|:-:|\n"
        f"| {m['Precisão']:.3f} | {m['Revocação']:.3f} | {m['mAP50']:.3f} | **{m['mAP50-95']:.3f}** "
        f"| {m['Inferência (ms)']:.2f} ms | {m['Parâmetros (M)']:.2f} M |"
    ))


def salvar_metricas(name, m):
    (METRICS_DIR / f"{name}.json").write_text(json.dumps(m, indent=2, ensure_ascii=False))
    mostrar_metricas(name, m)


def avaliar_ultralytics(name):
    data_yaml = DATA[name]
    split = split_avaliacao(data_yaml)
    best = RUNS_DIR / name / "weights" / "best.pt"
    model = YOLO(best)
    r = model.val(data=str(data_yaml), split=split, imgsz=IMGSZ, batch=BATCH,
                  project=str(RUNS_DIR), name=f"{name}_{split}", exist_ok=True,
                  plots=True, verbose=False)
    salvar_metricas(name, {
        "Split": split,
        "Precisão": float(r.box.mp),
        "Revocação": float(r.box.mr),
        "mAP50": float(r.box.map50),
        "mAP50-95": float(r.box.map),
        "Inferência (ms)": float(r.speed["inference"]),
        "Parâmetros (M)": sum(p.numel() for p in model.model.parameters()) / 1e6,
        "Arquivo (MB)": best.stat().st_size / 1e6,
    })


def mostrar_graficos(name):
    split = split_avaliacao(DATA[name])
    for pasta, arquivo, legenda in [
        (name, "results.png", "Curvas de treino"),
        (f"{name}_{split}", "confusion_matrix_normalized.png", "Matriz de confusão normalizada"),
    ]:
        p = RUNS_DIR / pasta / arquivo
        if p.exists():
            display(Markdown(f"**{legenda}**"))
            display(Image(filename=str(p), width=700))

<a id="yolov5"></a>
# 🟦 3. YOLOv5n

Treinado pelo **repositório original** [`ultralytics/yolov5`](https://github.com/ultralytics/yolov5) (`train.py` / `val.py`), com o dataset exportado no formato **YOLOv5 PyTorch**.

> ℹ️ O YOLOv5 clássico é *anchor-based*; os outros três são *anchor-free* e usam outro código de treino. A comparação mede o modelo **e** o pipeline de treino de cada geração.

In [ ]:
# @title 📥 Repositório do YOLOv5
# Já foi clonado (e as dependências instaladas) na seção 2.
YOLOV5_DIR = Path("/content/yolov5")
assert (YOLOV5_DIR / "train.py").exists(), "Rode a célula 'Instalar dependências' da seção 2."

In [ ]:
# @title ⬇️ Dataset no formato YOLOv5 PyTorch
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download("yolov5", location="/content/datasets/yolov5")

DATA["YOLOv5n"] = preparar_dataset(dataset.location)
resumo_dataset(DATA["YOLOv5n"])

In [ ]:
# @title 🏋️ Treinar YOLOv5n
V5_YAML = DATA["YOLOv5n"]
if not ja_treinado("YOLOv5n"):
    %cd {YOLOV5_DIR}
    !python train.py \
      --img {IMGSZ} \
      --batch {BATCH} \
      --epochs {EPOCHS} \
      --seed {SEED} \
      --data {V5_YAML} \
      --weights yolov5n.pt \
      --project {RUNS_DIR} \
      --name YOLOv5n \
      --exist-ok \
      --cache
    %cd /content

In [ ]:
# @title 📊 Avaliar YOLOv5n
# O val.py do YOLOv5 roda num processo separado para não misturar os módulos
# do repositório (models/, utils/) com o pacote ultralytics.
V5_EVAL = '''
import json, sys, torch
import val
data, weights, split, imgsz, batch, project = sys.argv[1:7]
(mp, mr, map50, map5095, *_), _, t = val.run(
    data=data, weights=weights, task=split, imgsz=int(imgsz), batch_size=int(batch),
    project=project, name="YOLOv5n_" + split, exist_ok=True, plots=True,
)
ckpt = torch.load(weights, map_location="cpu", weights_only=False)
n_params = sum(p.numel() for p in ckpt["model"].parameters())
print("@@JSON@@" + json.dumps({"mp": mp, "mr": mr, "map50": map50, "map": map5095,
                               "inf": t[1], "params": n_params}))
'''

split = split_avaliacao(V5_YAML)
best = RUNS_DIR / "YOLOv5n" / "weights" / "best.pt"
proc = subprocess.run(
    [sys.executable, "-c", V5_EVAL, str(V5_YAML), str(best), split, str(IMGSZ), str(BATCH), str(RUNS_DIR)],
    cwd=YOLOV5_DIR, capture_output=True, text=True,
)
saida = proc.stdout + proc.stderr
print("\n".join(l for l in saida.splitlines() if not l.startswith("@@JSON@@"))[-2500:])

linha = next((l for l in proc.stdout.splitlines() if l.startswith("@@JSON@@")), None)
if linha is None:
    raise RuntimeError("A avaliação do YOLOv5 falhou; veja a saída acima.")
r = json.loads(linha[len("@@JSON@@"):])
salvar_metricas("YOLOv5n", {
    "Split": split,
    "Precisão": r["mp"],
    "Revocação": r["mr"],
    "mAP50": r["map50"],
    "mAP50-95": r["map"],
    "Inferência (ms)": r["inf"],
    "Parâmetros (M)": r["params"] / 1e6,
    "Arquivo (MB)": best.stat().st_size / 1e6,
})

In [ ]:
# @title 📈 Gráficos do YOLOv5n
mostrar_graficos("YOLOv5n")

<a id="yolov8"></a>
# 🟩 4. YOLOv8n

Primeira geração do pacote `ultralytics`: cabeça *anchor-free* e desacoplada.

Treinado pelo pacote `ultralytics`, a partir de `yolov8n.pt`, com o dataset exportado no formato **`yolov8`**.

In [ ]:
# @title ⬇️ Dataset no formato yolov8
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download("yolov8", location="/content/datasets/yolov8")

DATA["YOLOv8n"] = preparar_dataset(dataset.location)
resumo_dataset(DATA["YOLOv8n"])

In [ ]:
# @title 🏋️ Treinar YOLOv8n
if not ja_treinado("YOLOv8n"):
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(DATA["YOLOv8n"]),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        deterministic=True,
        project=str(RUNS_DIR),
        name="YOLOv8n",
        exist_ok=True,
        plots=True,
    )

In [ ]:
# @title ↩️ Retomar treino do YOLOv8n (opcional)
# Se a sessão caiu NO MEIO do treino, rode esta célula para continuar do last.pt.
# YOLO(RUNS_DIR / "YOLOv8n" / "weights" / "last.pt").train(resume=True)

In [ ]:
# @title 📊 Avaliar YOLOv8n
avaliar_ultralytics("YOLOv8n")

In [ ]:
# @title 📈 Gráficos do YOLOv8n
mostrar_graficos("YOLOv8n")

<a id="yolo11"></a>
# 🟧 5. YOLO11n

Evolução do YOLOv8 com blocos C3k2 e C2PSA: menos parâmetros para a mesma precisão.

Treinado pelo pacote `ultralytics`, a partir de `yolo11n.pt`, com o dataset exportado no formato **`yolov11`**.

In [ ]:
# @title ⬇️ Dataset no formato yolov11
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download("yolov11", location="/content/datasets/yolov11")

DATA["YOLO11n"] = preparar_dataset(dataset.location)
resumo_dataset(DATA["YOLO11n"])

In [ ]:
# @title 🏋️ Treinar YOLO11n
if not ja_treinado("YOLO11n"):
    model = YOLO("yolo11n.pt")
    model.train(
        data=str(DATA["YOLO11n"]),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        deterministic=True,
        project=str(RUNS_DIR),
        name="YOLO11n",
        exist_ok=True,
        plots=True,
    )

In [ ]:
# @title ↩️ Retomar treino do YOLO11n (opcional)
# Se a sessão caiu NO MEIO do treino, rode esta célula para continuar do last.pt.
# YOLO(RUNS_DIR / "YOLO11n" / "weights" / "last.pt").train(resume=True)

In [ ]:
# @title 📊 Avaliar YOLO11n
avaliar_ultralytics("YOLO11n")

In [ ]:
# @title 📈 Gráficos do YOLO11n
mostrar_graficos("YOLO11n")

<a id="yolo12"></a>
# 🟥 6. YOLO12n

Arquitetura centrada em atenção (*area attention* e blocos R-ELAN). Suporte oficial na Ultralytics, sem fork e sem flash-attn.

Treinado pelo pacote `ultralytics`, a partir de `yolo12n.pt`, com o dataset exportado no formato **`yolov12`**.

In [ ]:
# @title ⬇️ Dataset no formato yolov12
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download("yolov12", location="/content/datasets/yolov12")

DATA["YOLO12n"] = preparar_dataset(dataset.location)
resumo_dataset(DATA["YOLO12n"])

In [ ]:
# @title 🏋️ Treinar YOLO12n
if not ja_treinado("YOLO12n"):
    model = YOLO("yolo12n.pt")
    model.train(
        data=str(DATA["YOLO12n"]),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        deterministic=True,
        project=str(RUNS_DIR),
        name="YOLO12n",
        exist_ok=True,
        plots=True,
    )

In [ ]:
# @title ↩️ Retomar treino do YOLO12n (opcional)
# Se a sessão caiu NO MEIO do treino, rode esta célula para continuar do last.pt.
# YOLO(RUNS_DIR / "YOLO12n" / "weights" / "last.pt").train(resume=True)

In [ ]:
# @title 📊 Avaliar YOLO12n
avaliar_ultralytics("YOLO12n")

In [ ]:
# @title 📈 Gráficos do YOLO12n
mostrar_graficos("YOLO12n")

<a id="comparacao"></a>
# 📊 7. Comparação

Junta as métricas salvas em `metricas/*.json` de todos os modelos já avaliados.

> ⚠️ **Sobre o `conf`:** o mAP é calculado com o limiar padrão (`conf=0.001`). Usar `conf=0.5` corta a curva precisão × revocação e dá um mAP distorcido. O limiar alto é para a **inferência** (seção 8), não para a métrica.

In [ ]:
# @title 🏆 Tabela de resultados
registros = {}
for name in MODEL_ORDER:
    f = METRICS_DIR / f"{name}.json"
    if f.exists():
        registros[name] = json.loads(f.read_text())
    else:
        print(f"⏭️  {name}: ainda não avaliado")

results_df = pd.DataFrame.from_dict(registros, orient="index")
results_df.index.name = "Modelo"
results_df.to_csv(OUT_DIR / "comparacao.csv")

quality = ["Precisão", "Revocação", "mAP50", "mAP50-95"]
cost    = ["Inferência (ms)", "Parâmetros (M)", "Arquivo (MB)"]
best_style = "background-color: #c8f7c5; color: #000; font-weight: bold"

display(
    results_df.style
    .format({**{c: "{:.3f}" for c in quality}, **{c: "{:.2f}" for c in cost}})
    .highlight_max(subset=quality, props=best_style)
    .highlight_min(subset=cost, props=best_style)
    .set_caption("Resultados por modelo (verde = melhor da coluna)")
    .set_table_styles([{"selector": "caption", "props": "font-size: 1.1em; font-weight: bold; padding: 8px;"}])
)

campeao = results_df["mAP50-95"].idxmax()
mais_rapido = results_df["Inferência (ms)"].idxmin()
display(Markdown(
    f"🏆 **Maior mAP50-95:** {campeao} ({results_df.loc[campeao, 'mAP50-95']:.3f})  \n"
    f"⚡ **Mais rápido:** {mais_rapido} ({results_df.loc[mais_rapido, 'Inferência (ms)']:.2f} ms/img)"
))

In [ ]:
# @title Gráfico de barras
names = results_df.index.tolist()
colors = [COLORS.get(n, "gray") for n in names]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
panels = [
    ("mAP50", "mAP@50", (0, 1.05)),
    ("mAP50-95", "mAP@50-95", (0, 1.05)),
    ("Inferência (ms)", "Inferência (ms/imagem) — menor é melhor", None),
]
for ax, (col, title, ylim) in zip(axes, panels):
    bars = ax.bar(names, results_df[col], color=colors, edgecolor="black", linewidth=0.6)
    ax.bar_label(bars, fmt="%.3f" if "mAP" in col else "%.2f", padding=3, fontsize=9)
    ax.set_title(title)
    if ylim:
        ax.set_ylim(*ylim)
    ax.grid(axis="x", visible=False)

fig.suptitle("Comparação dos modelos", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "comparacao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# @title Precisão × velocidade
fig, ax = plt.subplots(figsize=(8, 5))
for name, row in results_df.iterrows():
    ax.scatter(row["Inferência (ms)"], row["mAP50-95"], s=row["Parâmetros (M)"] * 120,
               color=COLORS.get(name, "gray"), alpha=0.8, edgecolor="black", zorder=3)
    ax.annotate(name, (row["Inferência (ms)"], row["mAP50-95"]),
                xytext=(8, 6), textcoords="offset points", fontweight="bold")
ax.set(xlabel="Inferência (ms/imagem)", ylabel="mAP@50-95",
       title="Precisão × velocidade (tamanho do ponto = nº de parâmetros)")
plt.tight_layout()
plt.savefig(OUT_DIR / "precisao_vs_velocidade.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# @title 📈 Curvas de treino lado a lado
def coluna(df, *opcoes):
    # O YOLOv5 original e o pacote ultralytics usam nomes de coluna diferentes.
    return next(c for c in opcoes if c in df.columns)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for name in MODEL_ORDER:
    csv = RUNS_DIR / name / "results.csv"
    if not csv.exists():
        continue
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    kw = dict(label=name, color=COLORS.get(name), lw=2)
    axes[0].plot(df["epoch"], df[coluna(df, "metrics/mAP50-95(B)", "metrics/mAP_0.5:0.95")], **kw)
    axes[1].plot(df["epoch"], df["train/box_loss"], **kw)
    axes[2].plot(df["epoch"], df["val/box_loss"], **kw)

for ax, t in zip(axes, ["mAP@50-95 (validação)", "Box loss — treino", "Box loss — validação"]):
    ax.set(title=t, xlabel="época")
    ax.legend()
fig.suptitle("Curvas de treino", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "curvas_treino.png", dpi=150, bbox_inches="tight")
plt.show()

<a id="inferencia"></a>
# 🔍 8. Inferência

Aqui sim usamos um limiar de confiança alto, para mostrar só as detecções em que o modelo confia. Por padrão, roda o modelo `ultralytics` com maior mAP50-95.

> ℹ️ Os pesos do YOLOv5 clássico não abrem com `YOLO(...)` do pacote `ultralytics`, por isso ele fica de fora desta seção. Para ele, use o `detect.py` do repositório.

In [ ]:
# @title Visualizar predições
CONF = 0.5      # @param {type:"slider", min:0.05, max:0.95, step:0.05}
N_IMAGENS = 6   # @param {type:"integer"}

candidatos = results_df.drop(index="YOLOv5n", errors="ignore")
INFER_MODEL = candidatos["mAP50-95"].idxmax()

root = DATA[INFER_MODEL].parent
source = root / ("test" if (root / "test" / "images").exists() else "valid") / "images"
model = YOLO(RUNS_DIR / INFER_MODEL / "weights" / "best.pt")

images = sorted(source.glob("*"))
images = images[:: max(1, len(images) // N_IMAGENS)][:N_IMAGENS]
preds = model.predict([str(p) for p in images], conf=CONF, imgsz=IMGSZ, verbose=False)

cols = 3
linhas = -(-len(preds) // cols)
fig, axes = plt.subplots(linhas, cols, figsize=(6 * cols, 4 * linhas), squeeze=False)
for ax, r in zip(axes.flat, preds):
    ax.imshow(r.plot()[..., ::-1])   # BGR → RGB
    ax.set_title(f"{len(r.boxes)} detecção(ões)", fontsize=10)
    ax.axis("off")
for ax in axes.flat[len(preds):]:
    ax.axis("off")
fig.suptitle(f"{INFER_MODEL} — imagens de teste (conf ≥ {CONF})", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# @title 💾 Salvar predições de todo o teste (opcional)
SALVAR_TUDO = False  # @param {type:"boolean"}
# Roda o modelo em TODAS as imagens de teste e salva as imagens anotadas + .txt
if SALVAR_TUDO:
    for _ in model.predict(source=str(source), conf=CONF, imgsz=IMGSZ, save=True, save_txt=True,
                           project=str(RUNS_DIR), name=f"{INFER_MODEL}_predict",
                           exist_ok=True, stream=True, verbose=False):
        pass
    display(Markdown(f"💾 Salvo em `{RUNS_DIR / f'{INFER_MODEL}_predict'}`"))

---

<div align="center">

Feito por **Davi Padula Rabelo** · treinado com [Ultralytics](https://docs.ultralytics.com) · dataset via [Roboflow](https://app.roboflow.com/workspace-hjntk/crossalkzebra-sdro0)

</div>